<a href="https://colab.research.google.com/github/aray-png/Problem-Set-2-Econ_3916_PS2.ipynb/blob/main/ps2_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS2 — The Survey That Wasn't Random

**Abigail Ray || ECON 3916 · Applied Data Analytics** · Topics 5-8 · 100 points · your role: Data Analyst, Growth team

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is data analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. You write the simulations, the bootstrap loop and the tests yourself. AI is authorised and required in Phase 4. If you cannot explain a line you submitted, you cannot submit it.

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Simulate the Population You Cannot See (30 pts)

### Step 1.1: Build a world with a known answer (15 pts)

In [3]:
rng = np.random.default_rng(42)

N_USERS = 50_000
BASE_BEFORE, BASE_AFTER = 3.432, 3.532      # true means 3.30 and 3.40

def simulate_population(base, n=N_USERS, rng=rng):
    """Return a DataFrame with engagement_minutes, engagement_z, had_support_ticket,
    latent s and the reported 1–5 satisfaction score."""

    engagement_minutes = rng.lognormal(mean=3.5, sigma=0.6, size=n)
    engagement_z = (engagement_minutes - engagement_minutes.mean()) / engagement_minutes.std()
    had_support_ticket = rng.random(n) < 0.22
    s = base + 0.35 * engagement_z - 0.60 * had_support_ticket + rng.normal(0, 0.75, size=n)
    satisfaction = np.clip(np.rint(s), 1, 5)

    return pd.DataFrame({
        "engagement_minutes": engagement_minutes,
        "engagement_z": engagement_z,
        "had_support_ticket": had_support_ticket,
        "s": s,
        "satisfaction": satisfaction
    })

before = simulate_population(BASE_BEFORE)
after = simulate_population(BASE_AFTER)
true_mean_before = before["satisfaction"].mean()
true_mean_after = after["satisfaction"].mean()
true_improvement = true_mean_after - true_mean_before

print(f"True mean satisfaction before: {true_mean_before:.3f}")
print(f"True mean satisfaction after:  {true_mean_after:.3f}")
print(f"True improvement:              {true_improvement:.3f}")

True mean satisfaction before: 3.291
True mean satisfaction after:  3.391
True improvement:              0.100


*Your answer here.*

The simulation creates a population of 50,000 users before and after the redesign, with satisfaction influenced by engagement minutes, support-ticket history, and random variation. The reported satisfaction score is rounded and restricted to a 1–5 scale. The true population mean is 3.291 before the redesign and 3.391 after, giving a true improvement of 0.100 points. This establishes the benchmark for evaluating the PM’s claim that any larger improvement observed in survey responses must be examined for potential bias rather than assumed to reflect a real change in population satisfaction.



### Step 1.2: Impose the response mechanism (15 pts)

In [4]:
def respond(df, a, b, rng=rng):
    """Who answers the survey: P(respond) = logistic(a + b*(s - 3.3) + 0.30*engagement_z
    - 0.25*had_support_ticket); a user answers when a uniform draw is below it."""

    z = (
        a
        + b * (df["s"] - 3.3)
        + 0.30 * df["engagement_z"]
        - 0.25 * df["had_support_ticket"]
    )

    p_respond = 1 / (1 + np.exp(-z))
    answered = rng.random(len(df)) < p_respond

    return answered

In [5]:

responded_before = respond(before, a=-1.0, b=0.45)
responded_after = respond(after, a=-2.75, b=2.00)
response_rate_before = responded_before.mean()
response_rate_after = responded_after.mean()

observed_mean_before = before.loc[responded_before, "satisfaction"].mean()
observed_mean_after = after.loc[responded_after, "satisfaction"].mean()

observed_gap = observed_mean_after - observed_mean_before

print(f"Pre-redesign response rate:  {response_rate_before:.3f}")
print(f"Post-redesign response rate: {response_rate_after:.3f}")
print(f"Observed pre-redesign mean:  {observed_mean_before:.3f}")
print(f"Observed post-redesign mean: {observed_mean_after:.3f}")
print(f"Apparent gap:                {observed_gap:.3f}")

Pre-redesign response rate:  0.275
Post-redesign response rate: 0.153
Observed pre-redesign mean:  3.592
Observed post-redesign mean: 4.309
Apparent gap:                0.717


> **🛑 Verification checkpoint.** You should get an observed pre-redesign mean of
> about **3.60** (accept 3.55 to 3.65) and an observed post-redesign mean of
> about **4.31** (accept 4.25 to 4.35) — an apparent gap of roughly **0.71**
> points out of a true improvement of **0.10**. Response rates should fall from
> roughly 27% to roughly 15%. If your numbers are outside those bands, your
> response function or your `base` values are wrong. **Do not proceed until it
> matches** — everything after this depends on reproducing the PM's slide.

*Your answer here.*

This is selection bias due to nonresponse, because the probability of answering depends on satisfaction, engagement, and support-ticket history. After the redesign, the response mechanism becomes much more selective toward highly satisfied users, causing the observed satisfaction mean to rise much more than the true population mean. The observed improvement is approximately 0.717 points, compared with the true improvement of 0.100 points. Response rates also fall from 27.5% before the redesign to 15.3% afterward. The survey reports a better satisfaction score despite hearing from fewer people. A larger survey would not fix this systematic bias because collecting more responses under the same selective mechanism reduces random sampling error but does not make respondents representative of the full population.

## Phase 2: Quantify What You Actually Know (35 pts)

### Step 2.1: Uncertainty without a formula (20 pts)

In [14]:

import numpy as np
from scipy import stats

def bootstrap_by_hand(x, stat, B=2000, rng=None):
    """Return B bootstrap statistics."""
    x = np.asarray(x)
    n = len(x)
    boot_stats = np.empty(B)

    for i in range(B):
        sample = rng.choice(x, size=n, replace=True)
        boot_stats[i] = stat(sample)

    return boot_stats

x = after.loc[responded_after, "satisfaction"].to_numpy()
rng = np.random.default_rng(101)

boot_mean = bootstrap_by_hand(x, np.mean, B=2000, rng=rng)
boot_median = bootstrap_by_hand(x, np.median, B=2000, rng=rng)

print("Mean percentile CI:", np.percentile(boot_mean, [2.5, 97.5]))
print("Median percentile CI:", np.percentile(boot_median, [2.5, 97.5]))
for name, stat in [("Mean", np.mean), ("Median", np.median)]:
    for method in ["percentile", "BCa"]:
        result = stats.bootstrap(
            (x,),
            stat,
            n_resamples=2000,
            confidence_level=0.95,
            method=method,
            rng=np.random.default_rng(101)
        )
        print(f"{name} {method} CI: {result.confidence_interval}")
        print("Unique bootstrap medians:", np.unique(boot_median))


Mean percentile CI: [4.29496923 4.32429955]
Median percentile CI: [4. 4.]
Mean percentile CI: ConfidenceInterval(low=np.float64(4.294969232783451), high=np.float64(4.3242995548572924))
Unique bootstrap medians: [4.]
Mean BCa CI: ConfidenceInterval(low=np.float64(4.2953429537008505), high=np.float64(4.324430479183032))
Unique bootstrap medians: [4.]
Median percentile CI: ConfidenceInterval(low=np.float64(4.0), high=np.float64(4.0))
Unique bootstrap medians: [4.]
Median BCa CI: ConfidenceInterval(low=np.float64(nan), high=np.float64(nan))
Unique bootstrap medians: [4.]


/usr/local/lib/python3.13/dist-packages/scipy/stats/_resampling.py:147: RuntimeWarning: invalid value encountered in scalar divide
  a_hat = 1/6 * sum(nums) / sum(dens)**(3/2)
/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:352: DegenerateDataWarning: The BCa confidence interval cannot be calculated. This problem is known to occur when the distribution is degenerate or the statistic is np.min.
  return fun(*args, **kwargs)


*Your answer here.*

The by hand and SciPy percentile intervals agree for both the mean and median. The mean percentile and BCa intervals are nearly identical. The median percentile interval is [4, 4] because every bootstrap resample has a median of 4. The median BCa interval returns NaN because the degenerate bootstrap distribution makes the BCa correction undefined. These intervals describe uncertainty among respondents and do not correct the selection bias caused by satisfaction dependent nonresponse.

### Step 2.2: Coverage is a claim you can check (15 pts)

In [15]:

def coverage_simulation(population, a, b, true_mean, R=50, B=100, rng=None):
    if rng is None:
        rng = np.random.default_rng(101)

    scores = population["satisfaction"].to_numpy()
    results = {"percentile": [], "BCa": []}

    for _ in range(R):
        answered = respond(population, a=a, b=b, rng=rng)
        x = scores[answered]

        if len(x) < 2:
            continue
        n = len(x)
        boot_means = np.empty(B)

        for j in range(B):
            boot_means[j] = np.mean(rng.choice(x, size=n, replace=True))


        low, high = np.percentile(boot_means, [2.5, 97.5])
        results["percentile"].append(low <= true_mean <= high)


        try:
            ci = stats.bootstrap(
                (x,),
                np.mean,
                n_resamples=B,
                confidence_level=0.95,
                method="BCa",
                rng=rng
            ).confidence_interval

            if np.isfinite(ci.low) and np.isfinite(ci.high):
                results["BCa"].append(ci.low <= true_mean <= ci.high)
        except (ValueError, ZeroDivisionError):
            pass

    for method, outcomes in results.items():
        outcomes = np.asarray(outcomes, dtype=float)
        n_valid = len(outcomes)

        if n_valid == 0:
            print(f"{method}: no valid intervals")
            continue

        coverage = outcomes.mean()
        mc_se = np.sqrt(coverage * (1 - coverage) / n_valid)

        print(
            f"{method}: coverage = {coverage:.3f}, "
            f"MC SE = {mc_se:.3f}, "
            f"valid intervals = {n_valid}/{R}"
        )


true_mean_after = after["satisfaction"].mean()

coverage_simulation(
    population=after,
    a=-2.75,
    b=2.00,
    true_mean=true_mean_after,
    R=200,
    B=500,
    rng=np.random.default_rng(101)
)

print(f"True population mean: {true_mean_after:.5f}")


percentile: coverage = 0.000, MC SE = 0.000, valid intervals = 200/200
BCa: coverage = 0.000, MC SE = 0.000, valid intervals = 200/200
True population mean: 3.39092


In [8]:
answered = respond(after, a=-2.75, b=2.00, rng=np.random.default_rng(101))
x = after["satisfaction"].to_numpy()[answered]

print("Sample size:", len(x))
print("Sample mean:", x.mean())
print("True mean:", true_mean_after)

for method in ["percentile", "BCa"]:
    ci = stats.bootstrap(
        (x,),
        np.mean,
        n_resamples=2000,
        confidence_level=0.95,
        method=method,
        rng=np.random.default_rng(101)
    ).confidence_interval

    print(method, ci)

Sample size: 7606
Sample mean: 4.310938732579542
True mean: 3.39092
percentile ConfidenceInterval(low=np.float64(4.295421377859585), high=np.float64(4.326324612148304))
BCa ConfidenceInterval(low=np.float64(4.2954246647383645), high=np.float64(4.326339108670472))


*Your answer here.*

Across 200 repeated surveys, both the percentile and BCa confidence intervals had 0% coverage, with a Monte Carlo standard error of 0.000 for each method. This means neither method produced an interval that contained the true population mean of 3.391 in any of the 200 simulations. The coverage difference between the methods was 0 percentage points, which is smaller than the Monte Carlo standard error, so there is no evidence that either method performs better. Both methods fail because people with higher satisfaction are more likely to respond, causing the confidence intervals to miss the true population mean. Increasing the number of bootstrap samples improves the stability of the intervals, but it does not correct this selection bias.

## Phase 3: Test Something Worth Testing (20 pts)

### Step 3.1: The test the PM wants

In [9]:

import numpy as np
from scipy import stats
from statsmodels.stats.power import TTestIndPower

x_before = before.loc[responded_before, "satisfaction"].to_numpy()
x_after = after.loc[responded_after, "satisfaction"].to_numpy()

test = stats.ttest_ind(x_after, x_before, equal_var=False)

mean_before = x_before.mean()
mean_after = x_after.mean()
observed_effect = mean_after - mean_before

pooled_sd = np.sqrt(
    ((len(x_before) - 1) * x_before.var(ddof=1)
     + (len(x_after) - 1) * x_after.var(ddof=1))
    / (len(x_before) + len(x_after) - 2)
)
cohens_d = observed_effect / pooled_sd

print("STEP 3.1: TWO-SAMPLE TEST")
print(f"Before mean:       {mean_before:.3f}")
print(f"After mean:        {mean_after:.3f}")
print(f"Observed effect:   {observed_effect:.3f} points")
print(f"Cohen's d:         {cohens_d:.3f}")
print(f"Welch t-statistic: {test.statistic:.3f}")
print(f"p-value:           {test.pvalue:.6g}")

STEP 3.1: TWO-SAMPLE TEST
Before mean:       3.592
After mean:        4.309
Observed effect:   0.717 points
Cohen's d:         0.879
Welch t-statistic: 66.331
p-value:           0


*Your answer here.*

The Welch two-sample t-test found a statistically significant difference between the pre- and post-redesign survey means (t = 66.331, p < 0.001). The observed mean increased from 3.592 to 4.309, a difference of 0.717 points. Cohen's d was 0.879, indicating a large standardized difference. However, the true population improvement was only 0.100 points. The main issue is selection bias, not statistical significance. Because people with higher satisfaction are more likely to respond after the redesign, the survey overstates the actual improvement. A significant p-value shows a difference between respondents' means, but it does not prove that the population experienced the same improvement.

### Step 3.2: Power, before the fact

In [10]:

power_analysis = TTestIndPower()

sd_before = before["satisfaction"].std(ddof=1)
sd_after = after["satisfaction"].std(ddof=1)
planning_sd = np.sqrt((sd_before**2 + sd_after**2) / 2)

target_effect = 0.10
standardized_effect = target_effect / planning_sd

required_n = power_analysis.solve_power(
    effect_size=standardized_effect,
    power=0.80,
    alpha=0.05,
    ratio=1.0,
    alternative="two-sided"
)
required_n = int(np.ceil(required_n))

n_before = len(x_before)
n_after = len(x_after)
ratio = n_after / n_before

detectable_d = power_analysis.solve_power(
    nobs1=n_before,
    alpha=0.05,
    power=0.80,
    ratio=ratio,
    alternative="two-sided"
)
detectable_points = detectable_d * planning_sd

print("\nSTEP 3.2: POWER ANALYSIS")
print(f"Planning SD:                    {planning_sd:.3f}")
print(f"Standardized effect for 0.10:   {standardized_effect:.3f}")
print(f"Required sample per group:      {required_n:,}")
print(f"Actual before respondents:      {n_before:,}")
print(f"Actual after respondents:       {n_after:,}")
print(f"Detectable effect (Cohen's d):  {detectable_d:.3f}")
print(f"Detectable effect in points:    {detectable_points:.3f}")

rng_power = np.random.default_rng(2026)
R = 2000
sample_sizes = [100, 250, 500, 1000, 2000]

true_improvement = (
    after["satisfaction"].mean() - before["satisfaction"].mean()
)

results = []
for m in sample_sizes:
    simulated_differences = np.empty(R)

    for r in range(R):
        xb = rng_power.choice(x_before, size=m, replace=True)
        xa = rng_power.choice(x_after, size=m, replace=True)
        simulated_differences[r] = xa.mean() - xb.mean()

    bias = simulated_differences.mean() - true_improvement
    rmse = np.sqrt(np.mean((simulated_differences - true_improvement)**2))

    results.append((m, simulated_differences.mean(), bias, rmse))

print("\nSIMULATION: DOES A LARGER SURVEY FIX THE BIAS?")
print("n per group | Mean estimated gap | Bias vs truth | RMSE")
for m, avg_gap, bias, rmse in results:
    print(f"{m:11d} | {avg_gap:18.3f} | {bias:13.3f} | {rmse:.3f}")



STEP 3.2: POWER ANALYSIS
Planning SD:                    0.887
Standardized effect for 0.10:   0.113
Required sample per group:      1,237
Actual before respondents:      13,759
Actual after respondents:       7,638
Detectable effect (Cohen's d):  0.040
Detectable effect in points:    0.035

SIMULATION: DOES A LARGER SURVEY FIX THE BIAS?
n per group | Mean estimated gap | Bias vs truth | RMSE
        100 |              0.720 |         0.621 | 0.631
        250 |              0.717 |         0.617 | 0.621
        500 |              0.716 |         0.617 | 0.619
       1000 |              0.717 |         0.617 | 0.618
       2000 |              0.717 |         0.617 | 0.617


*Your answer here.*

The power analysis shows that about 1,237 respondents per group are needed to detect a true 0.10-point improvement with 80% power. With the current sample sizes, the survey can detect differences as small as 0.035 points. The simulation shows that increasing the sample size does not fix selection bias. Even with 2,000 respondents per group, the estimated improvement stays around 0.717 points instead of the true 0.100 points. The bias remains around 0.617 points because more satisfied users are more likely to respond. A larger sample reduces random error but does not fix the biased results. The survey needs to address nonresponse instead of simply collecting more responses.

## Phase 4: AI Expansion — The Co-Pilot Era (15 pts)

**Status: AI usage AUTHORISED and REQUIRED.** You have built the simulation, the bootstrap and the power analysis by hand. Now use the **P.R.I.M.E. framework** to go further than you could in the time.

### Task 4.1: The weighting attempt (10 pts)

### AI Transcript

Paste **both prompt versions exactly as you sent them** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

AI TRANSCRIPT: CHAT GBT 4.0 October 24th.

My input:

[Prep] Act as a survey statistician who works on non-response
adjustment for product research.

[Request] I have a simulated user population with satisfaction scores
and two covariates, engagement_minutes and had_support_ticket, plus a
non-response mechanism where the probability of responding increases
with satisfaction. Write Python that implements inverse-probability
weighting to correct the observed survey mean, using a logistic model
of response on the covariates.

[Iterate] numpy, pandas, scikit-learn or statsmodels. Reuse my
variable names. Return the weighted estimate and its bootstrap
interval.

[Mechanism Check] Explain precisely which assumption makes this
correction valid, and what happens to it when response depends on
satisfaction ITSELF rather than only on the covariates.

[Evaluate] Given my mechanism, will this correction recover the true
mean? Answer yes or no and justify it.

AI's Response:
The key issue in your ECON 3916 simulation is that inverse-probability weighting (IPW) only corrects non-response bias when the response probability is adequately explained by the observed covariates. If response also depends directly on satisfaction, modeling response using only engagement_minutes and had_support_ticket will generally not recover the true population mean.
1. Copy-and-paste Python code
This code uses pandas, numpy, and scikit-learn to estimate response probabilities with logistic regression, calculate the IPW-adjusted mean, and construct a 95% bootstrap confidence interval.

It assumes your full simulated population is after, your response indicator is responded_after, and your satisfaction column is named s. Change "s" if your satisfaction column has a different name.

Important: This code is appropriate for evaluating your simulation because you have access to the full population. In a real survey, satisfaction for nonrespondents is generally unknown, so you could not directly calculate the true mean. Also, the bootstrap interval reflects uncertainty under this fitted model; it does not account for bias caused by the omitted satisfaction variable.
2. Mechanism check
The key assumption is missing at random (MAR), conditional on the observed covariates, together with positivity and a correctly specified response model.

Formally, the assumption is:
\[
R \perp S \mid X
\]

where:
- \(R\) = whether a user responds.
- \(S\) = satisfaction score.
- \(X\) = engagement_minutes and had_support_ticket.
This means that among users with the same covariate values, satisfaction must not influence their probability of responding.

If this assumption holds, the inverse response probability weights can correct for systematic differences between respondents and nonrespondents, provided every relevant type of user has a positive response probability and the model estimates those probabilities adequately.

If response depends on satisfaction itself even after conditioning on the covariates, the data are missing not at random (MNAR) under this model. The fitted logistic regression omits a predictor of response, and IPW using only those covariates generally cannot eliminate the resulting bias.
3. Will this correction recover the true mean?
No. Given your stated mechanism, response probability increases with satisfaction. Because the logistic model uses only engagement_minutes and had_support_ticket, it does not account for the direct relationship between satisfaction and response.

Consequently, higher-satisfaction users may remain overrepresented after weighting, leaving the estimated mean biased upward. The exact bias depends on the simulation parameters and how satisfaction relates to the covariates.

To correct this mechanism reliably, you would need additional information or assumptions, such as a justified model for the satisfaction-dependent response process, a validation sample of nonrespondents, or a sensitivity analysis. Simply adding satisfaction to the response model is not a practical solution when satisfaction is unobserved for nonrespondents.

My follow up entry:
Instead of using a covariate-only IPW method can you create an oracle benchmark that uses the true response probabilities from my previous simulation? Wouldnt this help test whether the remaining bias comes from the response model omitting satisfaction?

Chat GBT's response to my inquiry:

Yes — an oracle IPW benchmark is exactly the right diagnostic for your simulation. It lets you compare the covariate-only logistic model against the true response probabilities from your data-generating process and test whether misspecifying the response model explains the remaining bias.

The key distinction: because your response probability depends on satisfaction itself, even oracle IPW will only recover the true mean if the true probabilities are used correctly and the weighting assumptions are satisfied. In a simulation where the true response probabilities are known for every user, they can be used directly.
Copy-and-paste code
This assumes after contains the full population, responded_after is the response indicator, and respond() generated the response mechanism. You need to retain or reconstruct the true response probabilities from that mechanism; the code below assumes they are stored in true_p_response_after.
How to interpret the comparison
Result	Interpretation
Oracle bias is small; covariate-only bias is large	Evidence that the misspecified response model contributes to the bias.
Both estimates are biased similarly	Investigate the response mechanism, positivity, finite-sample variability, and weighting implementation.
Oracle estimate is close to truth, but truth falls outside its bootstrap interval	The estimate may be close by chance; investigate interval calibration and simulation variability.

One important correction to the earlier code: the bootstrap above resamples the full simulated population and keeps each user's true response probability attached to that user. This assesses sampling variability under the simulated population. For a complete assessment, you can also repeatedly regenerate the response indicators from the known probabilities to assess variability due to non-response itself.

Why oracle IPW works here
Let \(S_i\) be satisfaction and \(p_i=P(R_i=1\mid S_i,X_i)\) the true response probability. Then:
\[
E\left[\frac{R_iS_i}{p_i}\right]=E[S_i]
\]

because \(E[R_i\mid S_i,X_i]=p_i\). Thus, the oracle can correct satisfaction-dependent non-response, provided the true probabilities are correct and positive for every user. The normalized Hájek estimate is a ratio estimator, so it is not exactly unbiased in every finite sample, but it is consistent under standard regularity conditions.

Before running the code: if true_p_response_after was not saved by respond(), you will need to reconstruct it using the exact probability formula and parameters used in your simulation. Do not substitute the fitted logistic probabilities and label them oracle probabilities.



In [11]:
## CHAT GBT's response one:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

# Variables already defined in your simulation:
# after, responded_after, true_mean_after

df = after.copy()
df["responded_after"] = np.asarray(responded_after).astype(int)

covariates = ["engagement_minutes", "had_support_ticket"]
B = 1000
rng = np.random.default_rng(42)

def ipw_estimate(data):
    # Model response probability using covariates only
    X = data[covariates]
    y = data["responded_after"]

    model = LogisticRegression(max_iter=1000)
    model.fit(X, y)

    p_response = model.predict_proba(X)[:, 1]
    p_response = np.clip(p_response, 0.01, 1.0)

    # Keep respondents and weight by inverse response probability
    respondents = data["responded_after"].to_numpy() == 1
    satisfaction = data["s"].to_numpy()[respondents]
    weights = 1 / p_response[respondents]

    # Hájek normalized IPW estimate
    return np.average(satisfaction, weights=weights)

# Point estimate
weighted_mean = ipw_estimate(df)

# Bootstrap: resample population rows and refit the model
bootstrap_estimates = []

for _ in range(B):
    indices = rng.integers(0, len(df), size=len(df))
    sample = df.iloc[indices].reset_index(drop=True)

    if sample["responded_after"].nunique() < 2:
        continue

    try:
        bootstrap_estimates.append(ipw_estimate(sample))
    except (ValueError, ZeroDivisionError):
        continue

ci_lower, ci_upper = np.percentile(
    bootstrap_estimates, [2.5, 97.5]
)

print(f"True population mean: {true_mean_after:.4f}")
print(f"IPW weighted estimate: {weighted_mean:.4f}")
print(f"95% bootstrap interval: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"Bootstrap replicates used: {len(bootstrap_estimates)}")


True population mean: 3.3909
IPW weighted estimate: 4.1706
95% bootstrap interval: [4.1524, 4.1899]
Bootstrap replicates used: 1000


In [13]:
##Edited chat gbts rresponse 2 given my clarifying question
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

df = after.copy()
df["responded_after"] = np.asarray(responded_after).astype(int)

z = (
    -2.75
    + 2.00 * (df["s"] - 3.3)
    + 0.30 * df["engagement_z"]
    - 0.25 * df["had_support_ticket"].astype(float)
)

df["true_p_response"] = 1 / (1 + np.exp(-z))

covariates = ["engagement_minutes", "had_support_ticket"]
outcome = "satisfaction"
B = 1000
rng = np.random.default_rng(42)

def hajek_ipw(data, probabilities):
    responded = data["responded_after"].to_numpy().astype(bool)
    scores = data[outcome].to_numpy()[responded]
    weights = 1 / np.asarray(probabilities)[responded]
    return np.average(scores, weights=weights)

oracle_mean = hajek_ipw(df, df["true_p_response"].to_numpy())

model = LogisticRegression(max_iter=1000)
model.fit(df[covariates], df["responded_after"])
estimated_p = model.predict_proba(df[covariates])[:, 1]

covariate_mean = hajek_ipw(df, estimated_p)


oracle_boot = []
covariate_boot = []

for _ in range(B):
    sample = df.iloc[
        rng.integers(0, len(df), size=len(df))
    ].reset_index(drop=True)

    if sample["responded_after"].nunique() < 2:
        continue


    oracle_boot.append(
        hajek_ipw(sample, sample["true_p_response"].to_numpy())
    )

    boot_model = LogisticRegression(max_iter=1000)
    boot_model.fit(sample[covariates], sample["responded_after"])
    boot_p = boot_model.predict_proba(sample[covariates])[:, 1]

    covariate_boot.append(hajek_ipw(sample, boot_p))


true_mean = true_mean_after

print(f"True population mean: {true_mean:.4f}")
print(f"Oracle IPW estimate: {oracle_mean:.4f}")
print(f"Oracle bias: {oracle_mean - true_mean:.4f}")
print(
    "Oracle 95% bootstrap CI: "
    f"[{np.percentile(oracle_boot, 2.5):.4f}, "
    f"{np.percentile(oracle_boot, 97.5):.4f}]"
)

print(f"\nCovariate-only IPW estimate: {covariate_mean:.4f}")
print(f"Covariate-only bias: {covariate_mean - true_mean:.4f}")
print(
    "Covariate-only 95% bootstrap CI: "
    f"[{np.percentile(covariate_boot, 2.5):.4f}, "
    f"{np.percentile(covariate_boot, 97.5):.4f}]"
)

print(f"\nBootstrap replicates: {len(oracle_boot)}")


True population mean: 3.3909
Oracle IPW estimate: 3.4299
Oracle bias: 0.0390
Oracle 95% bootstrap CI: [3.3683, 3.4926]

Covariate-only IPW estimate: 4.1372
Covariate-only bias: 0.7463
Covariate-only 95% bootstrap CI: [4.1174, 4.1580]

Bootstrap replicates: 1000


### Task 4.2: The slide (5 pts)

*Your answer here.* #

**The Survey Wasn’t Random: Weighting Makes a Difference**

The post-redesign survey reports a satisfaction mean of 4.31, but the true population mean is 3.39. Respondents are disproportionately satisfied because satisfaction itself increases the likelihood of responding.

**What the survey supports:** Respondents report high satisfaction. However, the observed results do not reliably represent the entire user population.

**What weighting reveals:** Covariate-only inverse-probability weighting estimates satisfaction at 4.14 (95% bootstrap CI: 4.12–4.16), still substantially above the true mean. An oracle benchmark using the known response probabilities estimates 3.43 (95% bootstrap CI: 3.37–3.49), much closer to the truth. This comparison shows that omitting satisfaction from the response model contributes substantially to the remaining bias.

**Recommended change:** Randomly select initial nonrespondents for intensive follow-up and collect their satisfaction scores. This provides direct information about nonrespondents and helps assess and correct selection bias without relying entirely on unverifiable modeling assumptions.

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; both prompt versions are in it.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 2/Econ_3916_PS2.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.